In [1]:
import requests
from google.cloud import bigquery

Autenticação

In [2]:
client = bigquery.Client.from_service_account_json('C:/Users/ninem/Documents/storied-parser-332511-082bf009d3ff.json')

Criando Lista de ceps e endereços

In [5]:
lista_ceps = ["05778180", "13219816", "13214661", "04089001"]
lista_enderecos = []

Iterando os CEPs na requisição

In [6]:
for cep in lista_ceps:
    url: str = 'https://viacep.com.br/ws/{}/json'.format(cep)

    try:
        req = requests.get(url, timeout = 3)
        if req.status_code == 200:
            # API acessada com sucesso!
            endereco = req.json()
            lista_enderecos.append(
                            [
                                endereco['cep'],
                                endereco['logradouro'],
                                endereco['bairro'],
                                endereco['localidade'],
                                endereco['uf']
                            ]
            )

        else:
            erro = req.raise_for_status()
            print(f'Ocorreu o seguinte erro no acesso da API: {erro}')
    except Exception as erro:
         print(f'Ocorreu o seguinte erro na execução do código: {erro}')

for item in lista_enderecos:
    print(item)
    

['05778-180', 'Rua Caio Graco da Silva Prado', 'Parque Arariba', 'São Paulo', 'SP']
['13219-816', 'Rua Atibaia', 'Jardim Colônia', 'Jundiaí', 'SP']
['13214-661', 'Avenida Caetano Gornati', 'Engordadouro', 'Jundiaí', 'SP']
['04089-001', 'Alameda dos Maracatins', 'Indianópolis', 'São Paulo', 'SP']


Definindo as colunas que serão enviadas para o BigQuery

In [7]:
rows_to_insert = [{
    'cep': item[0],
    'logradouro': item[1],
    'bairro': item[2],
    'cidade': item[3],
    'estado': item[4]
}
for item in lista_enderecos]

Configurando o processo de envio dos dados e definindo o projeto, dataset e tabela do BQ

In [8]:
job_config = bigquery.LoadJobConfig(
    autodetect = True,
    write_disposition = "WRITE_APPEND"
)

table_id = "storied-parser-332511.General.enderecos"

Carregando os dados. usando load_table_from_json porque é o que funciona no free tier do BigQuery

In [9]:
load_job = client.load_table_from_json(rows_to_insert, table_id, job_config=job_config)
load_job.result()

print(f"{len(rows_to_insert)} registros carregados com sucesso!")

4 registros carregados com sucesso!
